# Step 3: SHAP and Model Contributions

This notebook calculates model-level contribution tables for the baseline XGBoost and Logistic Regression pipelines. The important correction in this version is that transformed columns are mapped back to the same original feature groups for both models, so one-hot encoded components are aggregated consistently.


## 1. Setup

Import packages, define project paths, and load the saved train/test split from Step 2.


In [ ]:
from pathlib import Path
import pickle
import time
import warnings

import joblib
import numpy as np
import pandas as pd
import shap

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 120)
pd.set_option("display.max_rows", 80)

PROJECT_ROOT = Path("/Users/yiqingwang/Documents/LLM_Exapliner")
COLAB_PROJECT_ROOT = Path("/content/drive/MyDrive/LLM_ClubLending")
BASE_DIR = COLAB_PROJECT_ROOT if COLAB_PROJECT_ROOT.exists() else PROJECT_ROOT

MODIFIED_DATA_DIR = BASE_DIR / "Data" / "ModifiedData"
BASELINE_MODEL_DIR = BASE_DIR / "Result" / "BaselineModel"
SHAP_DIR = BASE_DIR / "Result"  / "SHAP_retrained"
SHAP_DIR.mkdir(parents=True, exist_ok=True)

SPLIT_DATA_PATH = MODIFIED_DATA_DIR / "LC_split.pkl"
XGB_MODEL_PATH = BASELINE_MODEL_DIR / "xgb_pipeline_retrained.pkl"
LOGISTIC_MODEL_PATH = BASELINE_MODEL_DIR / "logistic_pipeline.pkl"

print(f"Base directory: {BASE_DIR}")
print(f"Split data path: {SPLIT_DATA_PATH}")
print(f"SHAP output directory: {SHAP_DIR}")


In [ ]:
if not SPLIT_DATA_PATH.exists():
    raise FileNotFoundError(f"Missing split data: {SPLIT_DATA_PATH}. Run Step 2 first.")

with open(SPLIT_DATA_PATH, "rb") as f:
    X_train, X_test, y_train, y_test = pickle.load(f)

# Keep row labels stable for downstream saved contribution files.
X_train = X_train.reset_index(drop=True)
y_train = y_train.reset_index(drop=True)
X_test = X_test.reset_index(drop=True)
y_test = y_test.reset_index(drop=True)

print(f"X_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")


## 2. Shared feature metadata

These lists must match Step 2. They define the original feature space that explanations should use after preprocessing.


In [ ]:
feature_name_map = {
    "home_ownership": "Home_Ownership_Status",
    "verification_status": "Income_Verification_Status",
    "purpose": "Loan_Purpose",
    "area": "Borrower_Area",
    "addr_state": "Borrower_State",
    "term": "Number_of_Payments",
    "grade": "Loan_Grade",
    "sub_grade": "Loan_Sub_Grade",
    "emp_length": "Employment_Length",
    "pub_rec_bankruptcies": "Number_of_Public_Bankruptcies",
    "funded_amnt": "Loan_Amount",
    "int_rate": "Interest_Rate",
    "installment": "Monthly_Payment",
    "annual_inc": "Annual_Income",
    "dti": "Debt_to_Income_Ratio",
    "delinq_2yrs": "Number_of_Delinquencies_in_Past_2_Years",
    "fico_range_low": "Lowest_FICO_Score",
    "fico_range_high": "Highest_FICO_Score",
    "inq_last_6mths": "Number_of_Inquiries_in_Last_6_Months",
    "open_acc": "Number_of_Open_Accounts",
    "pub_rec": "Number_of_Derogatory_Public_Records",
    "revol_bal": "Revolving_Balance",
    "revol_util": "Revolving_Utilization_Rate",
    "total_acc": "Total_Number_of_Accounts",
}
reverse_feature_map = {v: k for k, v in feature_name_map.items()}

nominal_cat_features = [
    "home_ownership",
    "verification_status",
    "purpose",
    "area",
    "addr_state",
]
ordinal_cat_features = [
    "term",
    "grade",
    "sub_grade",
    "emp_length",
    "pub_rec_bankruptcies",
]
num_features = [
    "funded_amnt",
    "int_rate",
    "installment",
    "annual_inc",
    "dti",
    "delinq_2yrs",
    "fico_range_low",
    "fico_range_high",
    "inq_last_6mths",
    "open_acc",
    "pub_rec",
    "revol_bal",
    "revol_util",
    "total_acc",
]
original_features = nominal_cat_features + ordinal_cat_features + num_features


## 3. Feature-name alignment helpers

The fitted `ColumnTransformer` expands nominal categorical fields into multiple one-hot columns, while ordinal and numeric fields remain one column each. These helpers create one authoritative mapping from original feature name to transformed-column positions. Both XGBoost SHAP and Logistic coefficients use this same mapping.


In [ ]:
def strip_transformer_prefix(feature_name):
    """Convert sklearn names like 'nominal__purpose_debt_consolidation' to 'purpose_debt_consolidation'."""
    return feature_name.split("__", 1)[1] if "__" in feature_name else feature_name


def get_transformed_feature_names(preprocessor):
    """Return transformed feature names in exactly the order produced by preprocessor.transform()."""
    return [strip_transformer_prefix(name) for name in preprocessor.get_feature_names_out()]


def build_feature_groups(preprocessor, original_feature_order, one_hot_features):
    """Map each original feature to transformed-column indices.

    This is the central alignment step. It handles:
    - one-hot columns for nominal features: home_ownership_RENT, purpose_credit_card, etc.
    - ordinal columns by exact name: term, grade, sub_grade, emp_length, pub_rec_bankruptcies
    - numeric columns by exact name: funded_amnt, int_rate, etc.

    Prefix matching is intentionally limited to nominal one-hot features. For example,
    pub_rec_bankruptcies starts with pub_rec_, but it is a separate ordinal feature and
    must not be grouped into pub_rec.
    """
    transformed_names = get_transformed_feature_names(preprocessor)
    one_hot_features = set(one_hot_features)
    groups = {}

    for feature in original_feature_order:
        exact_matches = [i for i, name in enumerate(transformed_names) if name == feature]
        one_hot_matches = []
        if feature in one_hot_features:
            one_hot_matches = [
                i
                for i, name in enumerate(transformed_names)
                if name.startswith(f"{feature}_")
            ]
        indices = exact_matches + one_hot_matches
        if not indices:
            raise ValueError(
                f"No transformed columns found for original feature '{feature}'. "
                "Check that Step 2 feature lists match the fitted pipeline."
            )
        groups[feature] = indices

    grouped_indices = sorted(index for indices in groups.values() for index in indices)
    if len(grouped_indices) != len(set(grouped_indices)):
        raise ValueError("A transformed column was assigned to more than one original feature.")

    missing_indices = sorted(set(range(len(transformed_names))) - set(grouped_indices))
    if missing_indices:
        missing_names = [transformed_names[i] for i in missing_indices]
        raise ValueError(f"Transformed columns were not assigned to any original feature: {missing_names}")

    return transformed_names, groups


def validate_feature_alignment(xgb_pipe, logistic_pipe):
    """Ensure XGBoost and Logistic pipelines expose the same transformed feature order and groups."""
    xgb_names, xgb_groups = build_feature_groups(
        xgb_pipe.named_steps["preprocess"], original_features, nominal_cat_features
    )
    log_names, log_groups = build_feature_groups(
        logistic_pipe.named_steps["preprocess"], original_features, nominal_cat_features
    )

    if xgb_names != log_names:
        raise ValueError("XGBoost and Logistic transformed feature names are not aligned.")
    if xgb_groups != log_groups:
        raise ValueError("XGBoost and Logistic feature-group mappings are not aligned.")

    print(f"Aligned transformed columns: {len(xgb_names)}")
    print(f"Aligned original feature groups: {len(xgb_groups)}")
    return xgb_names, xgb_groups


In [ ]:
def summarize_component_values(component_values, feature_groups):
    """Aggregate transformed-column values back to original feature-level summaries."""
    rows = []
    for feature, indices in feature_groups.items():
        signed_sum = float(np.sum(component_values[indices]))
        abs_sum = float(np.sum(np.abs(component_values[indices])))
        direction = 1 if signed_sum > 0 else -1 if signed_sum < 0 else 0
        rows.append({
            "Feature": feature,
            "SignedContribution": signed_sum,
            "ContributionDirection": direction,
            "AbsContribution": abs_sum,
            "NumTransformedColumns": len(indices),
        })
    return rows


def format_feature_values(row):
    formatted = {}
    for col, val in row.items():
        if pd.isna(val):
            formatted[col] = None
        elif col in ["int_rate", "revol_util"]:
            formatted[col] = round(float(val) * 100, 1)
        elif col == "annual_inc":
            formatted[col] = int(round(float(val)))
        elif col in num_features:
            formatted[col] = float(val) if isinstance(val, (np.floating, float)) else int(val) if isinstance(val, (np.integer, int)) else val
        else:
            formatted[col] = str(val)
    return formatted


def readable_feature_values(row):
    formatted = format_feature_values(row)
    return {feature_name_map[k]: v for k, v in formatted.items() if k in feature_name_map}


## 4. Load models and validate alignment

Before calculating any contribution table, confirm that both saved pipelines transform features into the same column order.


In [ ]:
if not XGB_MODEL_PATH.exists():
    raise FileNotFoundError(f"Missing XGBoost model: {XGB_MODEL_PATH}. Run Step 2 first.")
if not LOGISTIC_MODEL_PATH.exists():
    raise FileNotFoundError(f"Missing Logistic model: {LOGISTIC_MODEL_PATH}. Run Step 2 first.")

xgb_pipe = joblib.load(XGB_MODEL_PATH)
logistic_pipe = joblib.load(LOGISTIC_MODEL_PATH)

transformed_feature_names, feature_groups = validate_feature_alignment(xgb_pipe, logistic_pipe)

xgb_test_prob = xgb_pipe.predict_proba(X_test)[:, 1]
logistic_test_prob = logistic_pipe.predict_proba(X_test)[:, 1]


## 5. XGBoost SHAP values

Calculate SHAP values on the preprocessed matrix, then aggregate each row back to original feature names using the shared mapping above. For one-hot encoded variables, `AbsContribution` is the sum of absolute SHAP values across all encoded levels, while `SignedContribution` is the signed sum.


In [ ]:
def get_xgb_shap_for_rows(pipe, row_indices, X, y_prob, transformed_names, feature_groups, topk=None):
    preprocessor = pipe.named_steps["preprocess"]
    model = pipe.named_steps["model"]

    X_rows = X.loc[row_indices]
    X_rows_transformed = preprocessor.transform(X_rows)

    explainer = shap.TreeExplainer(model)
    shap_values = explainer.shap_values(X_rows_transformed)
    if isinstance(shap_values, list):
        shap_values = shap_values[1]

    records = []
    y_prob = pd.Series(y_prob, index=X.index)

    for position, row_index in enumerate(row_indices):
        original_row = X.loc[row_index]
        feature_values = format_feature_values(original_row)
        feature_summaries = summarize_component_values(shap_values[position], feature_groups)
        feature_summaries = sorted(feature_summaries, key=lambda row: row["AbsContribution"], reverse=True)
        if topk is not None:
            feature_summaries = feature_summaries[:topk]

        for rank, summary in enumerate(feature_summaries, start=1):
            feature = summary["Feature"]
            records.append({
                "Rowindex": row_index,
                "PredProb": float(y_prob.loc[row_index]),
                "FeatureRank": rank,
                "Feature": feature,
                "ReadableFeature": feature_name_map.get(feature, feature),
                "FeatureValue": feature_values.get(feature),
                "SignedContribution": round(summary["SignedContribution"], 6),
                "ContributionDirection": summary["ContributionDirection"],
                "AbsSHAPValue": round(summary["AbsContribution"], 6),
                "NumTransformedColumns": summary["NumTransformedColumns"],
            })

    return pd.DataFrame(records)


In [ ]:
# Set SAMPLE_ROWS to a small list for testing, or use list(X_test.index) for the full test set.
SAMPLE_ROWS = list(X_test.index)
BATCH_SIZE = 1000

xgb_results = []
batch = 0
for start in range(0, len(SAMPLE_ROWS), BATCH_SIZE):
    batch += 1
    batch_rows = SAMPLE_ROWS[start:start + BATCH_SIZE]
    batch_df = get_xgb_shap_for_rows(
        xgb_pipe,
        batch_rows,
        X_test,
        xgb_test_prob,
        transformed_feature_names,
        feature_groups,
        topk=None,
    )
    xgb_results.append(batch_df)

    batch_path = SHAP_DIR / f"xgb_shap_batch_{batch}.xlsx"
    batch_df.to_excel(batch_path, index=False)
    print(f"Saved {batch_path}")

xgb_shap_df = pd.concat(xgb_results, ignore_index=True)
xgb_final_path = SHAP_DIR / "XGBoost_SHAP_Aggregated.xlsx"
xgb_shap_df.to_excel(xgb_final_path, index=False)
print(f"Saved final XGBoost SHAP table to: {xgb_final_path}")


## 6. Logistic Regression contributions

For Logistic Regression, each transformed feature contribution is `transformed_value * coefficient` on the log-odds scale. The same feature-group mapping aggregates those transformed components back to original feature names.


In [ ]:
def calculate_logistic_contributions(pipe, X, y_prob, feature_groups):
    preprocessor = pipe.named_steps["preprocess"]
    model = pipe.named_steps["model"]

    X_transformed = preprocessor.transform(X)
    coef = model.coef_.ravel()
    intercept = float(model.intercept_.ravel()[0])

    # Works for sparse and dense transformed matrices.
    component_contrib = X_transformed.multiply(coef) if hasattr(X_transformed, "multiply") else X_transformed * coef
    if hasattr(component_contrib, "toarray"):
        component_contrib = component_contrib.toarray()

    logit_from_parts = intercept + component_contrib.sum(axis=1)
    prob_from_parts = 1 / (1 + np.exp(-logit_from_parts))
    max_error = np.max(np.abs(prob_from_parts - y_prob))
    print(f"Max probability reconstruction error: {max_error:.3e}")

    records = []
    for row_position, row_index in enumerate(X.index):
        original_row = X.loc[row_index]
        feature_values = format_feature_values(original_row)
        summaries = summarize_component_values(component_contrib[row_position], feature_groups)
        summaries = sorted(summaries, key=lambda row: row["AbsContribution"], reverse=True)

        for rank, summary in enumerate(summaries, start=1):
            feature = summary["Feature"]
            records.append({
                "Rowindex": row_index,
                "PredProb": float(y_prob[row_position]),
                "FeatureRank": rank,
                "Feature": feature,
                "ReadableFeature": feature_name_map.get(feature, feature),
                "FeatureValue": feature_values.get(feature),
                "SignedContribution": round(summary["SignedContribution"], 6),
                "ContributionDirection": summary["ContributionDirection"],
                "AbsContribution": round(summary["AbsContribution"], 6),
                "NumTransformedColumns": summary["NumTransformedColumns"],
            })

    return pd.DataFrame(records)


In [ ]:
logistic_contrib_df = calculate_logistic_contributions(
    logistic_pipe,
    X_test,
    logistic_test_prob,
    feature_groups,
)

logistic_long_path = SHAP_DIR / "Logistic_Contrib_Aggregated_Long.xlsx"
logistic_contrib_df.to_excel(logistic_long_path, index=False)
print(f"Saved Logistic contribution table to: {logistic_long_path}")


## 7. Optional wide Logistic contribution table

This preserves the older wide-table style, but the columns are now original feature names and therefore align with the XGBoost SHAP feature groups.


In [ ]:
def logistic_contributions_wide(pipe, X, feature_groups):
    preprocessor = pipe.named_steps["preprocess"]
    model = pipe.named_steps["model"]

    X_transformed = preprocessor.transform(X)
    coef = model.coef_.ravel()
    component_contrib = X_transformed.multiply(coef) if hasattr(X_transformed, "multiply") else X_transformed * coef
    if hasattr(component_contrib, "toarray"):
        component_contrib = component_contrib.toarray()

    wide = pd.DataFrame(index=X.index)
    for feature, indices in feature_groups.items():
        wide[feature] = component_contrib[:, indices].sum(axis=1)
    return wide


logistic_wide_df = logistic_contributions_wide(logistic_pipe, X_test, feature_groups)
logistic_wide_path = SHAP_DIR / "Logistic_Contrib.xlsx"
logistic_wide_df.to_excel(logistic_wide_path, index=True)
print(f"Saved wide Logistic contribution table to: {logistic_wide_path}")
